# PyDoseRT Publishing Plots (Patient Folder Source)

This notebook plots TPS vs PyDoseRT dose for one patient folder.

Expected input:
- `data/<patient>/pydosert_dose/dose_pred.npy` (generated by `examples/run_batch_dose_calc.ipynb`)

Outputs are saved under `out/publishing/<patient>/` by default.


In [ ]:
%matplotlib inline

from pathlib import Path
import sys
import numpy as np
import torch
from IPython.display import display, Image

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repo root (missing pyproject.toml/src).")

repo_root = find_repo_root(Path.cwd())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from pydosert.data import OptimizationConfig, loaders
from pydosert.utils.utils import find_patient_paths
from pydosert.utils.plotting import quick_plot, print_comparison_plot

print(f"Repo root: {repo_root}")

In [ ]:
# Optional helper: list available patients under data/
data_root = repo_root / "data"
patients = sorted([p.name for p in data_root.iterdir() if p.is_dir()]) if data_root.exists() else []
print(f"Found {len(patients)} patient folders:")
print(patients)

In [ ]:
# -------------------------------
# User controls (edit this cell)
# -------------------------------
PATIENT_DIR = repo_root / "data" / "0APUUTBFZ"

# Keep these aligned with test_vienna.py defaults for equivalent plots
STRUCT_NAMES = ["PTV", "Body"]
NEW_SPACING = (3.0, 3.0, 3.0)
CROP_VOLUME = False
USE_DELIVERY = True

OPTIMIZATION_JSON = repo_root / "src" / "pydosert" / "data" / "optimization_presets" / "vienna.json"
PRED_DOSE_PATH = PATIENT_DIR / "pydosert_dose" / "dose_pred.npy"

ISODOSE_PERCENT_LEVELS = (20, 40, 60, 80, 90, 95, 100, 105, 107, 110)
PROFILE_XLIM = (25, 150)
DOSE_CMAP = "turbo"
SHOW_CT_IN_QUICK_PLOT = False

SAVE_FIGURES = True
SHOW_INLINE_PLOTS = True
OUT_DIR = repo_root / "out" / "publishing" / PATIENT_DIR.name

print(f"PATIENT_DIR: {PATIENT_DIR}")
print(f"PRED_DOSE_PATH: {PRED_DOSE_PATH}")
print(f"OUT_DIR: {OUT_DIR}")

In [ ]:
if not PATIENT_DIR.exists():
    raise FileNotFoundError(f"Patient directory not found: {PATIENT_DIR}")
if not PRED_DOSE_PATH.exists():
    raise FileNotFoundError(f"Predicted dose file not found: {PRED_DOSE_PATH}")

ct_folder, rtplan_path, rtdose_path, rtstruct_path = find_patient_paths(PATIENT_DIR)

patient, _ = loaders.load_dicom(
    ct_folder=ct_folder,
    dose_path=rtdose_path,
    plan_path=rtplan_path,
    struct_path=rtstruct_path,
    new_spacing=NEW_SPACING,
    struct_names=STRUCT_NAMES,
    use_delivery=USE_DELIVERY,
    crop_volume=CROP_VOLUME,
    device="cpu",
)

# Match test_vienna.py behavior
if "Body" in patient.structures:
    patient.dose = patient.dose * patient.structures["Body"]

optimization = OptimizationConfig.from_json(OPTIMIZATION_JSON)
print("Loaded patient + optimization config.")
print(f"Dose tensor shape: {tuple(patient.dose.shape)}")
print(f"Available structures: {list(patient.structures.keys())}")

In [ ]:
dose_pred_np = np.load(PRED_DOSE_PATH)
dose_pred = torch.from_numpy(dose_pred_np).to(dtype=torch.float32)

if dose_pred.ndim == 4:
    dose_pred = dose_pred[0]

if dose_pred.shape != patient.dose.shape:
    raise ValueError(
        f"Shape mismatch: dose_pred={tuple(dose_pred.shape)} vs ref={tuple(patient.dose.shape)}"
    )

body_key = None
for key in patient.structures.keys():
    if key.lower() == "body" or "body" in key.lower():
        body_key = key
        break

if body_key is not None:
    body_mask = patient.structures[body_key]
    dose_pred = torch.where(body_mask, dose_pred, torch.zeros_like(dose_pred))
else:
    body_mask = torch.ones_like(patient.dose, dtype=torch.bool)
    print("Warning: no Body structure found; MAE uses full volume.")

mae_gy = torch.abs(dose_pred - patient.dose)[body_mask].mean().item()
mae_cgy = 100.0 * mae_gy
title = f"Patient {PATIENT_DIR.name}: MAE {mae_cgy:.2f} cGy"

print(f"Pred dose loaded: {PRED_DOSE_PATH}")
print(f"MAE: {mae_cgy:.2f} cGy")

In [ ]:
quick_out = OUT_DIR / f"quick_{PATIENT_DIR.name}.png"
comparison_out = OUT_DIR / f"comparison_{PATIENT_DIR.name}.png"

if SAVE_FIGURES:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

quick_plot(
    patient,
    dose_pred,
    title=title,
    show_ct=SHOW_CT_IN_QUICK_PLOT,
    out_path=str(quick_out) if SAVE_FIGURES else None,
)

print_comparison_plot(
    optimization,
    patient,
    dose_pred,
    out_path=str(comparison_out) if SAVE_FIGURES else None,
    isodose_percent_levels=ISODOSE_PERCENT_LEVELS,
    profile_xlim=PROFILE_XLIM,
    cmap_dose=DOSE_CMAP,
)

if SAVE_FIGURES:
    print(f"Saved quick plot: {quick_out}")
    print(f"Saved comparison plot: {comparison_out}")
    if SHOW_INLINE_PLOTS:
        display(Image(filename=str(quick_out)))
        display(Image(filename=str(comparison_out)))
else:
    print("Displayed plots inline (not saved).")